# Auto Generate Subtitled MKV from Video

In [ ]:
#@title Install Dependencies
!apt -y install ffmpeg
!pip install -q demucs openai-whisper yt-dlp

You can either directly translate via Whisper or use Google Translate in between. Results may vary

The code below downloads a video from YouTube, however you may also upload a file. Name it `video.webm`

In [ ]:
#@markdown Leave video_file as `video.webm` if you are using YouTube otherwise change it to your video file name
import subprocess
from pathlib import Path
import shutil

YT_URL = "" #@param {type:"string"}
VIDEO_FILE = "video.webm" #@param {type:"string"}

def download_video(url: str, output_name: str):
    if Path(output_name).exists():
        print(f"[!] File {output_name} already exists")
        return

    print(f"[!] Downloading video from {url}...")

    cmd = [
        "yt-dlp",
        "-f", "bestvideo[ext=webm]+bestaudio[ext=webm]/best[ext=webm]/best",
        "-o", output_name,
        url
    ]

    subprocess.run(cmd, check=True)

    print(f"[+] Downloaded to {output_name}")



In [ ]:
#@title Direct Translation (Whisper -> Target Language)
TARGET_LANG = "en" #@param {type:"string"}
DEMUX_AUDIO_FORMAT = "wav"
WHISPER_MODEL = "large" #@param {type:"string"}
DEVICE = "cuda" #@param ["cuda", "cpu"] {type:"string"}
CLEANUP_TEMP_FILES = True #@param {type:"boolean"}
WORKDIR = Path("/content/temp")
PROMPT = "" #@param {type:"string"}

def extract_audio(video_path: Path, audio_path: Path):
    cmd = [
        "ffmpeg",
        "-y",
        "-i", str(video_path),
        "-vn",
        "-acodec", "pcm_s16le",
        "-ar", "44100",
        "-ac", "2",
        str(audio_path)
    ]
    subprocess.run(cmd, check=True)
    print(f"[+] Extracted audio to {audio_path}")

def run_demucs(audio_path: Path, out_dir: Path):
    cmd = [
        "demucs",
        "--two-stems", "vocals",
        "--device", DEVICE,
        "-o", str(out_dir),
        str(audio_path)
    ]
    subprocess.run(cmd, check=True)
    filename = audio_path.stem
    vocals_path = out_dir / "htdemucs" / filename / "vocals.wav"
    print(f"[+] Demucs vocals saved at {vocals_path}")
    return vocals_path

def run_whisper(audio_path: Path, out_dir: Path, target_lang="en"):
    cmd = [
        "whisper",
        str(audio_path),
        "--model", WHISPER_MODEL,
        "--output_format", "srt",
        "--output_dir", str(out_dir),
        "--task", "translate" if target_lang != "auto" else "transcribe",
    ]
    if PROMPT:
        cmd.extend([
            "--initial_prompt", PROMPT,
            "--carry_initial_prompt", "True"
        ])
    if target_lang not in ["auto", "en"]:
        cmd.extend(["--language", target_lang])
    subprocess.run(cmd, check=True)
    srt_file = out_dir / f"{audio_path.stem}.srt"
    print(f"[+] Whisper generated SRT at {srt_file}")
    return srt_file

def mux_video_with_subtitle(video_path: Path, srt_path: Path, output_path: Path):
    cmd = [
        "ffmpeg",
        "-y",
        "-i", str(video_path),
        "-i", str(srt_path),
        "-c", "copy",
        "-c:s", "srt",
        "-map", "0:v",
        "-map", "0:a?",
        "-map", "1",
        "-metadata:s:s:0", "language=eng",
        str(output_path)
    ]
    subprocess.run(cmd, check=True)
    print(f"[+] Final MKV saved at {output_path}")

def main():
    workdir = WORKDIR
    workdir.mkdir(exist_ok=True, parents=True)

    video_path = Path(VIDEO_FILE)

    # 1. Download
    download_video(YT_URL, VIDEO_FILE)

    base_name = video_path.stem
    audio_path = workdir / f"{base_name}.{DEMUX_AUDIO_FORMAT}"
    demucs_out = workdir / "demucs_out"
    demucs_out.mkdir(exist_ok=True)
    srt_out = workdir / "srt_out"
    srt_out.mkdir(exist_ok=True)
    final_mkv = Path("/content") / f"{base_name}_final.mkv"

    # 2. Process
    extract_audio(video_path, audio_path)
    vocals_path = run_demucs(audio_path, demucs_out)
    print("[!] Generating subtitles (Whisper)...")
    srt_file = run_whisper(vocals_path, srt_out, target_lang=TARGET_LANG)
    mux_video_with_subtitle(video_path, srt_file, final_mkv)

    print("[+] All done!")
    if CLEANUP_TEMP_FILES:
      shutil.rmtree(workdir, ignore_errors=True)
      print("[+] Temp files deleted")

main()

In [ ]:
#@title Cascading Translation (Whisper -> SRT -> Google Translate -> Target Lang)
!pip install deep-translator
from deep_translator import GoogleTranslator


TARGET_LANG = "en" #@param {type:"string"}
DEMUX_AUDIO_FORMAT = "wav"
WHISPER_MODEL = "large" #@param {type:"string"}
DEVICE = "cuda" #@param ["cuda", "cpu"] {type:"string"}
WORKDIR = Path("/content/temp")


def extract_audio(video_path: Path, audio_path: Path):

    cmd = [
        "ffmpeg",
        "-y",
        "-i", str(video_path),
        "-vn",
        "-acodec", "pcm_s16le",
        "-ar", "44100",
        "-ac", "2",
        str(audio_path)
    ]

    subprocess.run(cmd, check=True)

    print(f"[+] Extracted audio to {audio_path}")


def run_demucs(audio_path: Path, out_dir: Path):

    cmd = [
        "demucs",
        "--two-stems", "vocals",
        "--device", DEVICE,
        "-o", str(out_dir),
        str(audio_path)
    ]

    subprocess.run(cmd, check=True)

    filename = audio_path.stem
    vocals_path = out_dir / "htdemucs" / filename / "vocals.wav"

    print(f"[+] Demucs vocals saved at {vocals_path}")

    return vocals_path


def run_whisper(audio_path: Path, out_dir: Path):

    cmd = [
        "whisper",
        str(audio_path),
        "--model", WHISPER_MODEL,
        "--output_format", "srt",
        "--output_dir", str(out_dir),
        "--task", "transcribe",
        "--verbose", "True"
    ]

    subprocess.run(cmd, check=True)

    srt_file = out_dir / f"{audio_path.stem}.srt"

    print(f"[+] Whisper generated SRT at {srt_file}")

    return srt_file


def translate_srt(input_srt: Path, output_srt: Path, target_lang="en"):

    translator = GoogleTranslator(source="auto", target=target_lang)

    with open(input_srt, "r", encoding="utf-8") as f:
        lines = f.readlines()

    translated_lines = []

    for line in lines:

        stripped = line.strip()

        if stripped.isdigit() or "-->" in stripped or stripped == "":
            translated_lines.append(line)
            continue

        try:
            translated = translator.translate(stripped)
            translated_lines.append(translated + "\n")

        except Exception:
            translated_lines.append(line)

    with open(output_srt, "w", encoding="utf-8") as f:
        f.writelines(translated_lines)

    print(f"[+] Translated subtitles saved at {output_srt}")

    return output_srt


def mux_video_with_subtitle(video_path: Path, srt_path: Path, output_path: Path):

    cmd = [
        "ffmpeg",
        "-y",
        "-i", str(video_path),
        "-i", str(srt_path),
        "-c", "copy",
        "-c:s", "srt",
        "-map", "0:v",
        "-map", "0:a?",
        "-map", "1",
        "-metadata:s:s:0", "language=eng",
        str(output_path)
    ]

    subprocess.run(cmd, check=True)

    print(f"[+] Final MKV saved at {output_path}")


def main():

    workdir = WORKDIR
    workdir.mkdir(exist_ok=True, parents=True)

    video_path = Path(VIDEO_FILE)

    download_video(YT_URL, VIDEO_FILE)

    base_name = video_path.stem

    audio_path = workdir / f"{base_name}.{DEMUX_AUDIO_FORMAT}"

    demucs_out = workdir / "demucs_out"
    demucs_out.mkdir(exist_ok=True)

    srt_out = workdir / "srt_out"
    srt_out.mkdir(exist_ok=True)

    final_mkv = Path("/content") / f"{base_name}_final.mkv"

    extract_audio(video_path, audio_path)

    vocals_path = run_demucs(audio_path, demucs_out)

    print("[!] Generating subtitles (Whisper)...")

    srt_file = run_whisper(vocals_path, srt_out)

    translated_srt = srt_out / "translated.srt"

    print("[!] Translating subtitles...")

    translate_srt(srt_file, translated_srt, TARGET_LANG)

    mux_video_with_subtitle(video_path, translated_srt, final_mkv)

    print("[+] All done!")

    shutil.rmtree(workdir, ignore_errors=True)

    print("[+] Temp files deleted")


main()